In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv("insurance.csv")

In [ ]:
df.head()

**EDA**

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
numeric_columns = ["age", "bmi", "children", "charges"]

for col in numeric_columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[col], kde=True, color="lightgreen" )
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()

In [ ]:
obj_column = ["sex","smoker","region"]

for col in obj_column:
    plt.figure(figsize=(8,4))
    sns.countplot(x=df[col], palette="rainbow")
    plt.title(f"Count of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.show()

In [ ]:
for col in numeric_columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=df[col], color="pink")
    plt.title(f"Boxplot of {col}")
    plt.xlabel(col)
    plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(numeric_only=True),annot =True, cmap="coolwarm", linewidths=0.5)
plt.title("Correlation Heatmap")
plt.show()

**Data Cleanig and Preprocessing**

In [ ]:
df_cleaned = df.copy()

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned.drop_duplicates(inplace=True)

In [ ]:
df_cleaned.shape

In [ ]:
df_cleaned.isnull().sum()

In [ ]:
df_cleaned.dtypes

In [ ]:
df_cleaned['sex'].value_counts()

In [ ]:
df_cleaned["sex"] = df_cleaned["sex"].map({"male":0, "female":1})

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned["smoker"] = df_cleaned["smoker"].map({"no":0, "yes":1})

In [ ]:
#df_cleaned["region"] = df_cleaned["region"].map({"northeast":0, "northwest":1, "southeast":2, "southwest":3})

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned.rename(columns={
    "sex": "is_female",
    "smoker": "is_smoker",
}, inplace=True)

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned = pd.get_dummies(df_cleaned, columns=["region"], drop_first=True)

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned = df_cleaned.astype(int)

In [ ]:
df_cleaned.head()

**Feature Engineering and Extraction**

In [ ]:
sns.histplot(df_cleaned["bmi"], kde=True, color="pink")

In [ ]:
df_cleaned["bmi_category"] = pd.cut(df_cleaned["bmi"], bins=[0, 18.5, 25, 30, np.inf], labels=["Underweight", "Normal weight", "Overweight", "Obese"])

In [ ]:
df_cleaned

In [ ]:
df_cleaned = pd.get_dummies(df_cleaned, columns=["bmi_category"], drop_first=True)

In [ ]:
df_cleaned 

In [ ]:
df_cleaned = df_cleaned.astype(int)

In [ ]:
df_cleaned.columns

In [ ]:
from sklearn.preprocessing import StandardScaler
cols = ["age", "bmi", "children"]
scaler = StandardScaler()
df_cleaned[cols] = scaler.fit_transform(df_cleaned[cols])

In [ ]:
df_cleaned.head()

In [ ]:
from scipy.stats import pearsonr


selected_features = [
    "age","bmi","children","is_female","is_smoker","region_northwest","region_southeast","region_southwest",
    "bmi_category_Normal weight","bmi_category_Overweight","bmi_category_Obese"
]

correlations = {
    feature: pearsonr(df_cleaned[feature], df_cleaned["charges"])[0] 
    for feature in selected_features
}

correlation_df = pd.DataFrame(list(correlations.items()), columns=["Feature", "Pearson Correlation"])
correlation_df.sort_values(by="Pearson Correlation", ascending=False)

In [ ]:
cat_features = [
    "is_female","is_smoker","region_northwest","region_southeast","region_southwest",
    "bmi_category_Normal weight","bmi_category_Overweight","bmi_category_Obese"
]



In [ ]:
from scipy.stats import chi2_contingency
import pandas as pd

alpha = 0.05

df_cleaned["charges_bin"] = pd.qcut(df_cleaned["charges"], q=4, labels=["Low", "Medium", "High", "Very High"])

chi2_results = {}

for col in cat_features:
    contingency_table = pd.crosstab(df_cleaned[col], df_cleaned["charges_bin"])
    chi2_stat, p_val, _, _ = chi2_contingency(contingency_table)
    decision = "Reject Null Hypothesis" if p_val < alpha else "Accept Null (Drop Feature)"
    chi2_results[col] = {"chi2_statistic": chi2_stat, "p_value": p_val, "decision": decision}

chi2_df = pd.DataFrame(chi2_results).T
chi2_df.sort_values(by="p_value")
chi2_df    

In [ ]:
final_df = df_cleaned[["age", "bmi", "children", "is_smoker", "charges","is_female", "region_northwest","region_southeast","region_southwest",
    "bmi_category_Normal weight","bmi_category_Overweight","bmi_category_Obese"]]

In [ ]:
final_df

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x = final_df.drop("charges", axis=1)
y = final_df["charges"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)   
 

In [ ]:
from sklearn.linear_model import LinearRegression


In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

In [ ]:
y_pred =  model.predict(X_test)

In [ ]:
from sklearn.metrics import r2_score
r2 = r2_score(y_test, y_pred)
n = X_test.shape[0]
p = X_test.shape[1]
adjusted_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
adjusted_r2